In [1]:
import numpy as np
import random
import math

RANDOM_SEED = 100
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# LOADING AND PREPARING THE DATASET

def read_names_from_file(filepath="/content/TrainingNames.txt"):
    # Reading raw text data
    with open(filepath, "r") as f:
        # skipping blank lines, stripping whitespace from each entry
        name_list = [line.strip() for line in f if line.strip()]
    return name_list

def create_character_vocab(name_list):
    # gathering every unique character that appears across all names
    all_chars = sorted(set("".join(name_list)))

    # special tokens go first: PAD fills unused positions,
    # SOS (Start of Sequence) / EOS (End of Sequence) mark the boundaries of each name during training
    all_chars = ["<PAD>", "<SOS>", "<EOS>"] + all_chars

    # Creating bidirectional mappings between characters and integer indices
    char_to_idx = {ch: idx for idx, ch in enumerate(all_chars)}
    idx_to_char = {idx: ch for ch, idx in char_to_idx.items()}

    return all_chars, char_to_idx, idx_to_char

def name_to_index_sequence(name, char_to_idx):
    # wrapping the character indices with start and end tokens so the model learns when a name begins and when to stop generating
    return (
        [char_to_idx["<SOS>"]]
        + [char_to_idx[ch] for ch in name if ch in char_to_idx]
        + [char_to_idx["<EOS>"]]
    )

# ACTIVATION FUNCTIONS AND BASIC MATH UTILS

def sigmoid_activation(x):
    # Formula: 1 / (1 + e^-x). Squashes values to [0, 1].
    # clipping to a safe range before passing to exp - avoids overflow for very large negative values
    return 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))

def tanh_activation(x):
    # Squashes values to [-1, 1], centered at zero. Standard for RNN hidden states.
    return np.tanh(x)

def softmax_activation(x):
    # Converts a vector of raw scores (logits) into a probability distribution that sums to 1.
    # subtracting the max value first for numerical stability (doesn't change the output but prevents exp from blowing up)
    shifted = np.exp(x - np.max(x))
    return shifted / shifted.sum()

def compute_cross_entropy(prob_dist, correct_idx):
    # Measures how far off our predicted probability is from the actual truth.
    # adds a tiny epsilon so we never take log(0), which would result in NaN
    return -math.log(prob_dist[correct_idx] + 1e-9)


# MODEL 1: VANILLA RNN

class VanillaRNN:
    """
    Architecture
    ────────────
    Single-layer Vanilla RNN for character-level language modelling.
      h_t = tanh( W_xh · x_t  +  W_hh · h_{t-1}  +  b_h )
      y_t = W_hy · h_t  +  b_y

    Parameters
    ──────────
    vocab_size  : V  (number of unique characters incl. special tokens)
    hidden_size : H  (number of hidden units)
    layers      : 1  (single recurrent layer)
    learning_rate : 0.001 (Adam)
    """

    def __init__(self, vocab_size, hidden_size=128, learning_rate=0.001):

        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.lr = learning_rate

        # small random init keeps activations near zero at the start, which helps with gradient flow through tanh
        scale = 0.01

        # Define the weight matrices and biases
        self.W_input_hidden = np.random.randn(hidden_size, vocab_size)  * scale  # input -> hidden
        self.W_hidden_hidden = np.random.randn(hidden_size, hidden_size) * scale  # recurrent connection
        self.bias_hidden = np.zeros((hidden_size, 1))

        self.W_hidden_output = np.random.randn(vocab_size, hidden_size)  * scale  # hidden -> output logits
        self.bias_output = np.zeros((vocab_size, 1))

        self._setup_adam_state()

    def _setup_adam_state(self):
        # Adam keeps a running estimate of the first and second moments for each parameter - initialise both to zero
        self.adam_m = {k: np.zeros_like(v) for k, v in self._get_params().items()}
        self.adam_v = {k: np.zeros_like(v) for k, v in self._get_params().items()}
        self.adam_step = 0  # counts how many updates we've done so far

    def _get_params(self):
        # Helper to easily grab all trainable parameters as a dictionary
        return {
            "W_xh": self.W_input_hidden,
            "W_hh": self.W_hidden_hidden,
            "b_h" : self.bias_hidden,
            "W_hy": self.W_hidden_output,
            "b_y" : self.bias_output,
        }

    def count_parameters(self):
        # Sums up all the elements in the weight matrices and biases
        return sum(p.size for p in self._get_params().values())

    def model_size_mb(self):
        # Each parameter is stored as a float64 (8 bytes). Convert to Megabytes.
        total_bytes = sum(p.nbytes for p in self._get_params().values())
        return total_bytes / (1024 ** 2)

    def report_size(self):
        # Just pretty-print the model's memory footprint
        params = self.count_parameters()
        size_mb = self.model_size_mb()
        print(f"\n  ── Vanilla RNN Size Report ──")
        print(f"  {'Parameter breakdown':}")
        for name, param in self._get_params().items():
            print(f"    {name:<6}: shape {str(param.shape):<20} → {param.size:>8,} params")
        print(f"  {'─'*45}")
        print(f"  Total trainable parameters : {params:>10,}")
        print(f"  Model size (float64)        : {size_mb:>10.4f} MB")

    def forward_pass(self, token_indices):
        # Running the RNN forward over a sequence of token indices
        # We need to save inputs, hidden states, and outputs to use later in backprop
        input_vecs  = {}
        hidden_states = {}
        output_logits = {}
        output_probs  = {}

        # hidden state before the first token is just zeros
        hidden_states[-1] = np.zeros((self.hidden_size, 1))
        total_loss = 0.0

        # at each step, predicting the next token given the current one
        for t, token_idx in enumerate(token_indices[:-1]):
            # one-hot encode current token
            input_vecs[t] = np.zeros((self.vocab_size, 1))
            input_vecs[t][token_idx] = 1

            # RNN update: combining input and previous hidden state
            hidden_states[t] = tanh_activation(
                self.W_input_hidden  @ input_vecs[t] +
                self.W_hidden_hidden @ hidden_states[t - 1] +
                self.bias_hidden
            )

            # Computing logits and converting to probabilities
            output_logits[t] = self.W_hidden_output @ hidden_states[t] + self.bias_output
            output_probs[t]  = softmax_activation(output_logits[t].ravel())

            # loss is negative log-likelihood of the correct next token
            total_loss += compute_cross_entropy(output_probs[t], token_indices[t + 1])

        return total_loss, input_vecs, hidden_states, output_probs

    def backward_pass(self, token_indices, input_vecs, hidden_states, output_probs):

        # initializing gradient accumulators for every weight matrix and bias
        dW_xh = np.zeros_like(self.W_input_hidden)
        dW_hh = np.zeros_like(self.W_hidden_hidden)
        db_h  = np.zeros_like(self.bias_hidden)
        dW_hy = np.zeros_like(self.W_hidden_output)
        db_y  = np.zeros_like(self.bias_output)

        # gradient that flows back from the next timestep
        grad_hidden_future = np.zeros((self.hidden_size, 1))

        num_steps = len(token_indices) - 1

        # Backpropagation Through Time (BPTT) - walk backwards through the sequence
        for t in reversed(range(num_steps)):
            # gradient of cross-entropy + softmax combined:
            # subtracting 1 from the probability of the correct class
            grad_output = output_probs[t].copy().reshape(-1, 1)
            grad_output[token_indices[t + 1]] -= 1

            # Accumulating gradients for the output layer
            dW_hy += grad_output @ hidden_states[t].T
            db_y  += grad_output

            # gradient flows back through the output weights AND from the future timesteps
            grad_hidden = self.W_hidden_output.T @ grad_output + grad_hidden_future

            # backprop through tanh: derivative of tanh(x) is (1 - tanh^2(x))
            grad_hidden_preact = (1 - hidden_states[t] ** 2) * grad_hidden

            # Accumulate gradients for the hidden and input layers
            db_h  += grad_hidden_preact
            dW_xh += grad_hidden_preact @ input_vecs[t].T
            dW_hh += grad_hidden_preact @ hidden_states[t - 1].T

            # passing gradient back to the previous timestep for the next iteration of the loop
            grad_hidden_future = self.W_hidden_hidden.T @ grad_hidden_preact

        gradients = {
            "W_xh": dW_xh, "W_hh": dW_hh, "b_h": db_h,
            "W_hy": dW_hy, "b_y": db_y
        }

        # gradient clipping - keeps training stable when gradients explode (common in vanilla RNNs)
        for key in gradients:
            np.clip(gradients[key], -5, 5, out=gradients[key])

        return gradients

    def _apply_adam(self, gradients, beta1=0.9, beta2=0.999, eps=1e-8):
        # Standard implementation of the Adam optimization algorithm
        self.adam_step += 1
        params = self._get_params()

        for key in gradients:

            # updating biased moment estimates (first and second moments)
            self.adam_m[key] = beta1 * self.adam_m[key] + (1 - beta1) * gradients[key]
            self.adam_v[key] = beta2 * self.adam_v[key] + (1 - beta2) * gradients[key] ** 2

            # correcting for the bias toward zero in early steps
            m_corrected = self.adam_m[key] / (1 - beta1 ** self.adam_step)
            v_corrected = self.adam_v[key] / (1 - beta2 ** self.adam_step)

            # Applying the update
            params[key] -= self.lr * m_corrected / (np.sqrt(v_corrected) + eps)

    def train_on_sequence(self, token_indices):
        # Wraps forward, backward, and optimization steps into a single call
        loss, input_vecs, hidden_states, output_probs = self.forward_pass(token_indices)
        gradients = self.backward_pass(token_indices, input_vecs, hidden_states, output_probs)
        self._apply_adam(gradients)
        return loss

    def generate_name(self, char_to_idx, idx_to_char, max_len=20, temperature=1.0):
        # Inference mode: generating a sequence character by character
        hidden = np.zeros((self.hidden_size, 1))

        # Starting with the SOS token
        current_idx = char_to_idx["<SOS>"]
        generated_chars = []

        for _ in range(max_len):

            # one-hot encoding the current character
            x = np.zeros((self.vocab_size, 1))
            x[current_idx] = 1

            # Computing next hidden state
            hidden = tanh_activation(
                self.W_input_hidden  @ x +
                self.W_hidden_hidden @ hidden +
                self.bias_hidden
            )

            # Computing logits
            logits = self.W_hidden_output @ hidden + self.bias_output

            # temperature scales the distribution:
            # > 1.0 increases randomness, < 1.0 makes it more conservative/confident
            probs = softmax_activation(logits.ravel() / temperature)

            # Sample from the distribution instead of just taking the argmax
            current_idx = np.random.choice(len(probs), p=probs)

            # Stopping if the model predicts the end of the sequence
            if current_idx == char_to_idx["<EOS>"]:
                break

            # Appending valid characters to our result
            if current_idx not in (char_to_idx["<PAD>"], char_to_idx["<SOS>"]):
                generated_chars.append(idx_to_char[current_idx])

        return "".join(generated_chars)


# MODEL 2: BIDIRECTIONAL LSTM (BLSTM)

class LSTMCell:
    # Single LSTM cell (no batching) designed to handle one timestep

    def __init__(self, input_size, hidden_size, scale=0.01):

        H, I = hidden_size, input_size
        # packing all four gate weight matrices into a single matrix for efficiency:
        # rows 0:H -> input gate, H:2H -> forget gate, 2H:3H -> cell gate, 3H:4H -> output gate
        # This allows us to compute all gate pre-activations in one matrix multiplication
        self.W = np.random.randn(4 * H, I + H) * scale
        self.b = np.zeros((4 * H, 1))
        self.H = H

    def forward(self, x, h_prev, c_prev):
        # concatenate input and previous hidden state before multiplying
        xh    = np.vstack([x, h_prev])
        gates = self.W @ xh + self.b
        H     = self.H

        # splitting the gate pre-activations and apply the right nonlinearity to each
        input_gate  = sigmoid_activation(gates[:H])          # how much new info to write
        forget_gate = sigmoid_activation(gates[H:2 * H])     # how much old memory to keep
        cell_gate   = tanh_activation(gates[2 * H:3 * H])    # candidate new memory content
        output_gate = sigmoid_activation(gates[3 * H:])      # how much of the cell to expose

        # Updating the long-term cell state
        cell_state  = forget_gate * c_prev + input_gate * cell_gate

        # Calculating the short-term hidden state
        hidden_state = output_gate * tanh_activation(cell_state)

        # storing everything needed for the backward pass (cache)
        cache = (x, h_prev, c_prev, input_gate, forget_gate, cell_gate,
                 output_gate, cell_state, hidden_state, xh)

        return hidden_state, cell_state, cache

    def backward(self, grad_hidden, grad_cell, cache):
        # Unpacking the cache from the forward pass
        x, h_prev, c_prev, i_g, f_g, c_g, o_g, cell, hidden, xh = cache
        H = self.H

        # backprop through output gate and tanh(cell)
        d_out_gate  = grad_hidden * tanh_activation(cell)
        d_cell_total = grad_cell + grad_hidden * o_g * (1 - tanh_activation(cell) ** 2)

        # backprop into each gate
        d_forget = d_cell_total * c_prev
        d_input  = d_cell_total * c_g
        d_cgate  = d_cell_total * i_g
        d_cell_prev = d_cell_total * f_g

        # backprop through the gate nonlinearities (derivatives of sigmoid and tanh)
        d_input_raw  = d_input  * i_g * (1 - i_g)
        d_forget_raw = d_forget * f_g * (1 - f_g)
        d_cgate_raw  = d_cgate  * (1 - c_g ** 2)
        d_out_raw    = d_out_gate * o_g * (1 - o_g)

        # Stack them back up to match the fused weight matrix structure
        d_gates = np.vstack([d_input_raw, d_forget_raw, d_cgate_raw, d_out_raw])

        # Gradients w.r.t weights and biases
        dW      = d_gates @ xh.T
        db      = d_gates
        d_xh    = self.W.T @ d_gates

        # Split the gradient w.r.t xh back into x and h_prev
        dx       = d_xh[:x.shape[0]]
        d_h_prev = d_xh[x.shape[0]:]

        return dx, d_h_prev, d_cell_prev, dW, db


class BidirectionalLSTM:
    """
    Architecture
    ────────────
    Bidirectional LSTM: one forward LSTM + one backward LSTM.
    The hidden states from both directions are concatenated and
    projected to the vocabulary for next-character prediction
    (using the FORWARD direction's output as the generative head).
      Forward  : h_t^f, c_t^f = LSTM_f(x_t, h_{t-1}^f, c_{t-1}^f)
      Backward : h_t^b, c_t^b = LSTM_b(x_t, h_{t+1}^b, c_{t+1}^b)
      concat_t = [h_t^f ; h_t^b]   (2H dim)
      y_t      = W_out · concat_t + b_out

    Parameters
    ──────────
    vocab_size    : V
    hidden_size   : H  (each direction has H units → 2H combined)
    layers        : 1 (per direction)
    learning_rate : 0.001 (Adam)
    """

    def __init__(self, vocab_size, hidden_size=128, learning_rate=0.001):

        self.vocab_size  = vocab_size
        self.hidden_size = hidden_size
        self.lr          = learning_rate

        # separating LSTM cells for left-to-right and right-to-left passes
        self.forward_cell  = LSTMCell(vocab_size, hidden_size)
        self.backward_cell = LSTMCell(vocab_size, hidden_size)

        scale = 0.01
        # output layer takes the concatenated forward+backward hidden state (which is why it's 2 * hidden_size)
        self.W_out = np.random.randn(vocab_size, 2 * hidden_size) * scale
        self.b_out = np.zeros((vocab_size, 1))

        self._setup_adam_state()

    def _all_weights(self):
        # flat list of all trainable arrays - used by Adam
        return [
            self.forward_cell.W,  self.forward_cell.b,
            self.backward_cell.W, self.backward_cell.b,
            self.W_out, self.b_out
        ]

    def count_parameters(self):
        return sum(p.size for p in self._all_weights())

    def _setup_adam_state(self):
        self.adam_m    = [np.zeros_like(p) for p in self._all_weights()]
        self.adam_v    = [np.zeros_like(p) for p in self._all_weights()]
        self.adam_step = 0

    def _make_one_hot(self, idx):
        # Utility to generate one-hot vectors for a given token
        vec = np.zeros((self.vocab_size, 1))
        vec[idx] = 1
        return vec

    def forward_pass(self, token_indices):

        num_steps = len(token_indices) - 1
        input_vecs = [self._make_one_hot(token_indices[t]) for t in range(num_steps)]

        # left-to-right pass (process sequence from start to end)
        hf, cf = np.zeros((self.hidden_size, 1)), np.zeros((self.hidden_size, 1))
        fwd_hiddens, fwd_caches = [], []

        for t in range(num_steps):
            hf, cf, cache = self.forward_cell.forward(input_vecs[t], hf, cf)
            fwd_hiddens.append(hf)
            fwd_caches.append(cache)

        # right-to-left pass - start from the end of the sequence and move backward
        hb, cb = np.zeros((self.hidden_size, 1)), np.zeros((self.hidden_size, 1))
        bwd_hiddens = [None] * num_steps
        bwd_caches  = [None] * num_steps

        for t in reversed(range(num_steps)):
            hb, cb, cache = self.backward_cell.forward(input_vecs[t], hb, cb)
            bwd_hiddens[t] = hb
            bwd_caches[t]  = cache

        # combining both directions and compute output probabilities
        total_loss = 0.0
        output_probs = []
        for t in range(num_steps):

            # Vertical stack concatenates the features of both directions
            combined = np.vstack([fwd_hiddens[t], bwd_hiddens[t]])
            logits   = self.W_out @ combined + self.b_out
            probs    = softmax_activation(logits.ravel())

            output_probs.append(probs)
            total_loss += compute_cross_entropy(probs, token_indices[t + 1])

        return total_loss, input_vecs, fwd_hiddens, bwd_hiddens, fwd_caches, bwd_caches, output_probs

    def backward_pass(self, token_indices, input_vecs, fwd_hiddens, bwd_hiddens,
                       fwd_caches, bwd_caches, output_probs):

        num_steps = len(token_indices) - 1

        # Initialize gradient placeholders
        dW_out = np.zeros_like(self.W_out)
        db_out = np.zeros_like(self.b_out)
        dW_fwd = np.zeros_like(self.forward_cell.W)
        db_fwd = np.zeros_like(self.forward_cell.b)
        dW_bwd = np.zeros_like(self.backward_cell.W)
        db_bwd = np.zeros_like(self.backward_cell.b)

        # gradients that propagate across timesteps for each direction
        grad_hf_next = np.zeros((self.hidden_size, 1))
        grad_cf_next = np.zeros((self.hidden_size, 1))
        grad_hb_next = np.zeros((self.hidden_size, 1))
        grad_cb_next = np.zeros((self.hidden_size, 1))

        # We step backwards through the combined outputs
        for t in reversed(range(num_steps)):

            grad_out = output_probs[t].copy().reshape(-1, 1)
            grad_out[token_indices[t + 1]] -= 1

            combined = np.vstack([fwd_hiddens[t], bwd_hiddens[t]])
            dW_out += grad_out @ combined.T
            db_out += grad_out

            # splitting the gradient back into the two directions from the output weights
            grad_combined = self.W_out.T @ grad_out
            grad_hf = grad_combined[:self.hidden_size]  + grad_hf_next
            grad_hb = grad_combined[self.hidden_size:]  + grad_hb_next

            # Route gradients to respective LSTM cells
            _, grad_hf_next, grad_cf_next, dWf, dbf = self.forward_cell.backward(grad_hf, grad_cf_next, fwd_caches[t])
            _, grad_hb_next, grad_cb_next, dWb, dbb = self.backward_cell.backward(grad_hb, grad_cb_next, bwd_caches[t])

            # Accumulate weight gradients
            dW_fwd += dWf; db_fwd += dbf
            dW_bwd += dWb; db_bwd += dbb

        gradients = [dW_fwd, db_fwd, dW_bwd, db_bwd, dW_out, db_out]

        # Applying gradient clipping to avoid explosion
        for g in gradients:
            np.clip(g, -5, 5, out=g)

        return gradients

    def _apply_adam(self, gradients, beta1=0.9, beta2=0.999, eps=1e-8):
        self.adam_step += 1
        weights = self._all_weights()

        for i, (w, g) in enumerate(zip(weights, gradients)):

            self.adam_m[i] = beta1 * self.adam_m[i] + (1 - beta1) * g
            self.adam_v[i] = beta2 * self.adam_v[i] + (1 - beta2) * g ** 2
            m_hat = self.adam_m[i] / (1 - beta1 ** self.adam_step)
            v_hat = self.adam_v[i] / (1 - beta2 ** self.adam_step)
            w -= self.lr * m_hat / (np.sqrt(v_hat) + eps)

    def train_on_sequence(self, token_indices):
        result = self.forward_pass(token_indices)
        loss   = result[0]
        grads  = self.backward_pass(token_indices, *result[1:])
        self._apply_adam(grads)

        return loss

    def generate_name(self, char_to_idx, idx_to_char, max_len=20, temperature=1.0):
        # at generation time we only have the forward direction - the backward hidden state is fixed at zero since there's no future context available
        hf, cf = np.zeros((self.hidden_size, 1)), np.zeros((self.hidden_size, 1))
        hb_zero = np.zeros((self.hidden_size, 1))

        current_idx = char_to_idx["<SOS>"]
        generated_chars = []

        for _ in range(max_len):

            x = np.zeros((self.vocab_size, 1))
            x[current_idx] = 1

            # Step forward
            hf, cf, _ = self.forward_cell.forward(x, hf, cf)

            # Concatenate with the empty backward state
            combined = np.vstack([hf, hb_zero])
            logits   = self.W_out @ combined + self.b_out
            probs    = softmax_activation(logits.ravel() / temperature)

            # Sample next character
            current_idx = np.random.choice(len(probs), p=probs)
            if current_idx == char_to_idx["<EOS>"]:
                break
            if current_idx not in (char_to_idx["<PAD>"], char_to_idx["<SOS>"]):
                generated_chars.append(idx_to_char[current_idx])

        return "".join(generated_chars)

# MODEL 3: RNN WITH ATTENTION

class RNNWithAttention:
    """
    Architecture
    ────────────
    Encoder-Decoder RNN with additive (Bahdanau-style) attention.
    Encoder (1-layer Vanilla RNN):
        h_t^enc = tanh( W_xe · x_t  +  W_he · h_{t-1}^enc  +  b_e )

    Attention (applied at each decoder step s):
        score_t = v^T · tanh( W_a · h_t^enc  +  U_a · h_s^dec )
        α_t     = softmax(scores)
        context = Σ_t α_t · h_t^enc

    Decoder (1-layer Vanilla RNN):
        h_s^dec = tanh( W_xd · [x_s ; context]  +  W_hd · h_{s-1}^dec  +  b_d )
        y_s     = W_out · h_s^dec  +  b_out

    Parameters
    ──────────
    vocab_size    : V
    hidden_size   : H
    layers        : 1 (encoder) + 1 (decoder)
    learning_rate : 0.001 (Adam)
    """

    def __init__(self, vocab_size, hidden_size=128, learning_rate=0.001):

        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.lr = learning_rate
        s = 0.01

        # encoder RNN weights
        self.W_enc_input  = np.random.randn(hidden_size, vocab_size)  * s
        self.W_enc_hidden = np.random.randn(hidden_size, hidden_size) * s
        self.b_enc = np.zeros((hidden_size, 1))

        # attention mechanism weights (Bahdanau / additive style)
        # W_attn and U_attn project encoder and decoder states into a shared space, then v_attn scores how relevant each encoder position is
        self.W_attn = np.random.randn(hidden_size, hidden_size) * s
        self.U_attn = np.random.randn(hidden_size, hidden_size) * s
        self.v_attn = np.random.randn(hidden_size, 1)            * s

        # decoder RNN weights - input is [current token; context vector] concatenated
        self.W_dec_input  = np.random.randn(hidden_size, vocab_size + hidden_size) * s
        self.W_dec_hidden = np.random.randn(hidden_size, hidden_size)              * s
        self.b_dec        = np.zeros((hidden_size, 1))

        # final projection from decoder hidden state to vocabulary logits
        self.W_output = np.random.randn(vocab_size, hidden_size) * s
        self.b_output = np.zeros((vocab_size, 1))

        self._setup_adam_state()

    def _all_weights(self):
        # Used by the Adam optimizer to iterate over all parameters
        return [
            self.W_enc_input, self.W_enc_hidden, self.b_enc,
            self.W_attn, self.U_attn, self.v_attn,
            self.W_dec_input, self.W_dec_hidden, self.b_dec,
            self.W_output, self.b_output
        ]

    def count_parameters(self):
        return sum(p.size for p in self._all_weights())

    def _setup_adam_state(self):
        self.adam_m    = [np.zeros_like(p) for p in self._all_weights()]
        self.adam_v    = [np.zeros_like(p) for p in self._all_weights()]
        self.adam_step = 0

    def _make_one_hot(self, idx):
        vec = np.zeros((self.vocab_size, 1))
        vec[idx] = 1
        return vec

    def _run_encoder(self, token_indices):
        # running a vanilla RNN over the input tokens and collect every hidden state - the decoder will attend over this full sequence of states
        enc_hidden_states = []
        h = np.zeros((self.hidden_size, 1))

        for idx in token_indices:
            x = self._make_one_hot(idx)
            h = tanh_activation(self.W_enc_input @ x + self.W_enc_hidden @ h + self.b_enc)
            enc_hidden_states.append(h)

        return enc_hidden_states

    def _compute_attention(self, enc_hidden_states, dec_hidden):
        # scoring each encoder position against the current decoder state
        scores = []

        for h_enc in enc_hidden_states:
            # Score formula: v^T * tanh(W*h_enc + U*h_dec)
            energy = self.v_attn.T @ tanh_activation(
                self.W_attn @ h_enc + self.U_attn @ dec_hidden
            )
            scores.append(energy[0, 0])

        # turning scores into a probability distribution over encoder positions (alpha weights)
        attention_weights = softmax_activation(np.array(scores))

        # context vector is a weighted sum of encoder hidden states based on the computed alphas
        context = sum(a * h for a, h in zip(attention_weights, enc_hidden_states))

        return context, attention_weights

    def forward_pass(self, token_indices):

        num_steps = len(token_indices) - 1

        # encoding everything except the last token to form the memory bank
        enc_hidden_states = self._run_encoder(token_indices[:-1])

        dec_hidden = np.zeros((self.hidden_size, 1))
        total_loss = 0.0

        # Lists to cache intermediate variables for the backward pass
        all_probs, all_contexts, all_alphas, all_dec_hiddens, all_dec_inputs = [], [], [], [], []

        for t in range(num_steps):
            # attend over encoder states to get a context vector specific to this decoder step
            context, alpha = self._compute_attention(enc_hidden_states, dec_hidden)

            # feeding current token and context together into the decoder
            dec_input = np.vstack([self._make_one_hot(token_indices[t]), context])

            # Decoder step
            dec_hidden = tanh_activation(
                self.W_dec_input  @ dec_input  +
                self.W_dec_hidden @ dec_hidden +
                self.b_dec
            )

            # Map to vocabulary probabilities
            logits = self.W_output @ dec_hidden + self.b_output
            probs  = softmax_activation(logits.ravel())

            total_loss += compute_cross_entropy(probs, token_indices[t + 1])

            # Cache everything
            all_probs.append(probs)
            all_contexts.append(context)
            all_alphas.append(alpha)
            all_dec_hiddens.append(dec_hidden)
            all_dec_inputs.append(dec_input)

        return total_loss, enc_hidden_states, all_probs, all_contexts, all_alphas, all_dec_hiddens, all_dec_inputs

    def backward_pass(self, token_indices, enc_hidden_states, all_probs, all_contexts,
                      all_alphas, all_dec_hiddens, all_dec_inputs):

        num_steps = len(token_indices) - 1

        # zero out gradient buffers for every weight matrix
        dW_ei = np.zeros_like(self.W_enc_input); dW_eh = np.zeros_like(self.W_enc_hidden); db_e = np.zeros_like(self.b_enc)
        dW_a  = np.zeros_like(self.W_attn); dU_a  = np.zeros_like(self.U_attn);       dv_a = np.zeros_like(self.v_attn)
        dW_di = np.zeros_like(self.W_dec_input);    dW_dh = np.zeros_like(self.W_dec_hidden); db_d = np.zeros_like(self.b_dec)
        dW_out = np.zeros_like(self.W_output); db_out = np.zeros_like(self.b_output)

        grad_dec_hidden_future = np.zeros((self.hidden_size, 1))

        # accumulating gradients w.r.t. each encoder hidden state
        grad_enc_states = [np.zeros((self.hidden_size, 1)) for _ in enc_hidden_states]

        # Step backward through the decoder
        for t in reversed(range(num_steps)):

            grad_out = all_probs[t].copy().reshape(-1, 1)
            grad_out[token_indices[t + 1]] -= 1

            dW_out  += grad_out @ all_dec_hiddens[t].T
            db_out  += grad_out

            grad_dec_h = self.W_output.T @ grad_out + grad_dec_hidden_future
            grad_dec_preact = (1 - all_dec_hiddens[t] ** 2) * grad_dec_h

            dW_di += grad_dec_preact @ all_dec_inputs[t].T
            dW_dh += grad_dec_preact @ (all_dec_hiddens[t - 1].T if t > 0 else np.zeros((1, self.hidden_size)))
            db_d  += grad_dec_preact

            # Passing gradient back in time for the decoder
            grad_dec_hidden_future = self.W_dec_hidden.T @ grad_dec_preact

            # approximating attention gradient - treat alpha weights as fixed constants (straight-through estimator for the softmax in attention)
            # We slice the decoder input weights to only grab the part corresponding to the context vector
            grad_context = self.W_dec_input[:, self.vocab_size:].T @ grad_dec_preact

            # Distributing the context gradient back to the encoder states based on the attention weights
            for k, (alpha_k, h_enc_k) in enumerate(zip(all_alphas[t], enc_hidden_states)):
                grad_enc_states[k] += alpha_k * grad_context

        # backprop through the encoder RNN
        grad_enc_h = np.zeros((self.hidden_size, 1))

        for t in reversed(range(len(enc_hidden_states))):

            # Adding gradient coming from the attention mechanism
            grad_enc_h += grad_enc_states[t]

            x = self._make_one_hot(token_indices[t])
            grad_enc_preact = (1 - enc_hidden_states[t] ** 2) * grad_enc_h

            dW_ei += grad_enc_preact @ x.T
            dW_eh += grad_enc_preact @ (enc_hidden_states[t - 1].T if t > 0 else np.zeros((1, self.hidden_size)))
            db_e  += grad_enc_preact

            # Passing gradient back in time for the encoder
            grad_enc_h = self.W_enc_hidden.T @ grad_enc_preact

        gradients = [dW_ei, dW_eh, db_e, dW_a, dU_a, dv_a, dW_di, dW_dh, db_d, dW_out, db_out]

        # Applying clipping to all gradients
        for g in gradients:
            np.clip(g, -5, 5, out=g)

        return gradients

    def _apply_adam(self, gradients, beta1=0.9, beta2=0.999, eps=1e-8):
        # Updating rule for Adam optimizer applied across all 11 parameter matrices
        self.adam_step += 1
        weights = self._all_weights()

        for i, (w, g) in enumerate(zip(weights, gradients)):
            self.adam_m[i] = beta1 * self.adam_m[i] + (1 - beta1) * g
            self.adam_v[i] = beta2 * self.adam_v[i] + (1 - beta2) * g ** 2

            m_hat = self.adam_m[i] / (1 - beta1 ** self.adam_step)
            v_hat = self.adam_v[i] / (1 - beta2 ** self.adam_step)

            w -= self.lr * m_hat / (np.sqrt(v_hat) + eps)

    def train_on_sequence(self, token_indices):
        result    = self.forward_pass(token_indices)
        loss      = result[0]
        gradients = self.backward_pass(token_indices, *result[1:])
        self._apply_adam(gradients)

        return loss

    def generate_name(self, char_to_idx, idx_to_char, max_len=20, temperature=1.0):
        # encoding just the <SOS> token to initialise the encoder context
        # (only one state at first; we grow it as we generate each character)
        seed_tokens    = [char_to_idx["<SOS>"]]
        enc_hidden_states = self._run_encoder(seed_tokens)

        dec_hidden  = np.zeros((self.hidden_size, 1))
        current_idx = char_to_idx["<SOS>"]
        generated_chars = []

        for _ in range(max_len):
            # Computing context vector
            context, _ = self._compute_attention(enc_hidden_states, dec_hidden)

            # Concatenating token input and context
            dec_input  = np.vstack([self._make_one_hot(current_idx), context])

            # Stepping the decoder forward
            dec_hidden = tanh_activation(
                self.W_dec_input  @ dec_input  +
                self.W_dec_hidden @ dec_hidden +
                self.b_dec
            )

            # Getting probabilities
            logits = self.W_output @ dec_hidden + self.b_output
            probs  = softmax_activation(logits.ravel() / temperature)

            # Sample next character
            current_idx = np.random.choice(len(probs), p=probs)
            if current_idx == char_to_idx["<EOS>"]:
                break

            if current_idx not in (char_to_idx["<PAD>"], char_to_idx["<SOS>"]):
                generated_chars.append(idx_to_char[current_idx])

                # appending the decoder's hidden state to the encoder context so future attention steps can look back at already-generated chars
                enc_hidden_states.append(dec_hidden)

        return "".join(generated_chars)

# TRAINING LOOP

def run_training(model, encoded_sequences, epochs=30, print_every=5):

    epoch_losses = []
    for epoch in range(1, epochs + 1):
        # shuffle every epoch so the model doesn't memorise sequence order
        random.shuffle(encoded_sequences)
        total_loss = 0.0

        for seq in encoded_sequences:
            if len(seq) < 2:
                continue  # skipping degenerate sequences
            total_loss += model.train_on_sequence(seq)

        avg_loss = total_loss / len(encoded_sequences)
        epoch_losses.append(avg_loss)

        if epoch % print_every == 0:
            print(f"  Epoch {epoch:3d}/{epochs}  | Loss: {avg_loss:.4f}")

    return epoch_losses


# TASK 2 - QUANTITATIVE EVALUATION

def evaluate_model(model, char_to_idx, idx_to_char, training_names_set,
                   n_samples=200, temperature=0.8):

    generated_names = []

    for _ in range(n_samples):
        name = model.generate_name(char_to_idx, idx_to_char,
                                   max_len=25, temperature=temperature)
        if name.strip():
            generated_names.append(name.strip())

    # novelty: fraction of generated names that don't appear in training data
    novelty_rate = (
        sum(1 for n in generated_names if n not in training_names_set)
        / len(generated_names)
        if generated_names else 0
    )

    # diversity: how many of the generated names are actually unique among themselves
    unique_count = len(set(generated_names))
    diversity    = unique_count / len(generated_names) if generated_names else 0

    return {
        "n_generated"  : len(generated_names),
        "novelty_rate" : novelty_rate,
        "diversity"    : diversity,
        "unique_names" : unique_count,
        "samples"      : generated_names[:10],
    }

def print_model_summary(label, model, hidden_size, lr):
    # Printing a nice summary box with model metadata
    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"{'='*60}")
    print(model.__doc__)
    print(f"  Hidden size   : {hidden_size}")
    print(f"  Learning rate : {lr}")
    print(f"  Trainable params: {model.count_parameters():,}")
    print(f"{'='*60}")


# Execution starts here
print("Loading dataset …")
names = read_names_from_file("TrainingNames.txt")
print(f"  {len(names)} names loaded.")

all_chars, char_to_idx, idx_to_char = create_character_vocab(names)
vocab_size = len(all_chars)
print(f"  Vocabulary size: {vocab_size}")

encoded_names = [name_to_index_sequence(n, char_to_idx) for n in names]
training_names_set = set(names)

HIDDEN_SIZE = 256
LEARN_RATE  = 0.001
NUM_EPOCHS  = 50

# Model 1: Vanilla RNN
rnn = VanillaRNN(vocab_size, hidden_size=HIDDEN_SIZE, learning_rate=LEARN_RATE)
print_model_summary("Model 1: Vanilla RNN", rnn, HIDDEN_SIZE, LEARN_RATE)

# reporting parameter count and model size for the Vanilla RNN
rnn.report_size()

print("\nTraining …")
run_training(rnn, encoded_names, epochs=NUM_EPOCHS)

# Model 2: Bidirectional LSTM
blstm = BidirectionalLSTM(vocab_size, hidden_size=HIDDEN_SIZE, learning_rate=LEARN_RATE)
print_model_summary("Model 2: Bidirectional LSTM", blstm, HIDDEN_SIZE, LEARN_RATE)
print("\nTraining …")
run_training(blstm, encoded_names, epochs=NUM_EPOCHS)

# Model 3: RNN + Attention
attn_rnn = RNNWithAttention(vocab_size, hidden_size=HIDDEN_SIZE, learning_rate=LEARN_RATE)
print_model_summary("Model 3: RNN with Attention", attn_rnn, HIDDEN_SIZE, LEARN_RATE)
print("\nTraining …")
run_training(attn_rnn, encoded_names, epochs=NUM_EPOCHS)

# Task 2: Evaluation
print("\n\n" + "="*60)
print("  TASK 2 - QUANTITATIVE EVALUATION")
print("="*60)

all_results = {}
for label, model in [("Vanilla RNN", rnn), ("BLSTM", blstm), ("RNN+Attention", attn_rnn)]:

    results = evaluate_model(model, char_to_idx, idx_to_char, training_names_set, n_samples=200, temperature=0.8)
    all_results[label] = results

    print(f"\n── {label} ──")
    print(f"  Generated       : {results['n_generated']}")
    print(f"  Novelty Rate    : {results['novelty_rate']*100:.1f}%  "
            f"(names not in training set)")
    print(f"  Diversity       : {results['diversity']*100:.1f}%  "
            f"({results['unique_names']} unique / {results['n_generated']} total)")
    print(f"  Sample names    : {', '.join(results['samples'])}")

# Comparison table
print("\n\n" + "="*60)
print("  COMPARISON TABLE")
print("="*60)
print(f"{'Model':<20} {'Novelty Rate':>14} {'Diversity':>12} {'Params':>12}")
print("-"*60)

for label, model in [("Vanilla RNN", rnn), ("BLSTM", blstm), ("RNN+Attention", attn_rnn)]:

    r = all_results[label]
    print(f"{label:<20} {r['novelty_rate']*100:>13.1f}% "
            f"{r['diversity']*100:>11.1f}% "
            f"{model.count_parameters():>12,}")

print("="*60)

Loading dataset …
  1000 names loaded.
  Vocabulary size: 55

  Model 1: Vanilla RNN

    Architecture
    ────────────
    Single-layer Vanilla RNN for character-level language modelling.

      h_t = tanh( W_xh · x_t  +  W_hh · h_{t-1}  +  b_h )
      y_t = W_hy · h_t  +  b_y

    Parameters
    ──────────
    vocab_size  : V  (number of unique characters incl. special tokens)
    hidden_size : H  (number of hidden units)
    layers      : 1  (single recurrent layer)
    learning_rate : 0.001 (Adam)
    
  Hidden size   : 256
  Learning rate : 0.001
  Trainable params: 94,007

  ── Vanilla RNN Size Report ──
  Parameter breakdown
    W_xh  : shape (256, 55)            →   14,080 params
    W_hh  : shape (256, 256)           →   65,536 params
    b_h   : shape (256, 1)             →      256 params
    W_hy  : shape (55, 256)            →   14,080 params
    b_y   : shape (55, 1)              →       55 params
  ─────────────────────────────────────────────
  Total trainable parameter

**Task 3: Qualitative Analysis**

**Realism of Generated Names**

Vanilla RNN & BLSTM:

These models demonstrate a strong grasp of realism. They successfully generate culturally accurate, well-formed Indian first names and surnames. Examples like "Shreya Agarwal", "Brijesh Soni", "Dhruv Siddiqui", and "Rajesh Rathore" look like genuine names that could easily belong to real people.

RNN+Attention:

This model fails the realism test. While it achieves a 100% novelty rate, it does so by generating absolute gibberish. Names like "Vivekh Mopep", "Vikrash Gopop", and "Sitra Mishao" do not align with any recognizable Indian naming conventions and sound entirely synthetic.

**Common Failure Modes**

1. Mode Collapse and Repetition

    * The RNN+Attention model suffers from severe mode collapse. It gets stuck generating variations of the same prefix (e.g., "Vikra", "Vikrash", "Vikran", "Vikrah"). Also, we can see that it outputs the exact same name ("Vikra More") three times. This failure is reflected quantitatively in its low diversity score of just 63.0%.

    * The Vanilla RNN also exhibits a mild form of exact-match repetition, outputting "Pankaj Alva" twice in its 10-name sample, though its overall diversity remains high at 84.5%.

2. Truncation / Cut-off Generation

    * The BLSTM model struggles slightly with word boundaries, sometimes terminating generation before a longer surname is fully resolved. For example, "Shyam Balakrish" appears abruptly truncated and is likely missing a common suffix like "-nan" or "-na".

3. Nonsensical Syllable Composition (Over-creativity)

    * Because the RNN+Attention model forces a 100% novelty rate, it ends up inventing phonetic combinations that don't exist in the target distribution. Surnames like "Mopep", "Gopop", and "Morep" highlight the model's inability to learn valid character transitions for this dataset.

    * The BLSTM shows very minor signs of this as well, generating slightly atypical surnames like "Senku" or phonetic misspellings like "Dus" (which should likely be "Das").